# AI Investment Committee

AAI-520 Final Team Project, Group 7: Michael Valderrama, Guna Pasupathy, Mina Habib.

Repository: https://github.com/ibucketbranch/AAI-520-Final-Project

This is the skeleton the final notebook gets assembled into. Each numbered section maps to a graded item in docs/RUBRIC_MAP.md and names its owner. Owners replace their TODO cells with working demos; the section order stays.

This project is a course demonstration and not financial advice.

## 1. Agent design and workflows

A research run starts with a stock symbol. The planner orders three research steps (earnings, news, market) and reads the memory note left by the previous run for the same symbol, moving a source up the order when that note flagged a gap there. Each planned step is routed to its gatherer, and everything the gatherers return lands in one pool of typed Evidence rows in SQLite. News goes through more steps: raw articles go through ingest, preprocess, classify, extract, and summarize, and only articles that pass the classification gate become Evidence. Two analyst agents, Bull and Bear, then draft claims from the pool, with every claim citing the evidence ids it rests on. An evaluator judges each claim against the full text of its cited evidence; unsupported claims go back to their side with the reason attached, and the loop stops when everything is supported or the round cap is hit. One chair call over the surviving claims issues the verdict, the memo is written, and a memory note (confidence plus open gaps) is saved for the next run to read.

The three graded workflow patterns and where this notebook shows them: prompt chaining is the news pipeline in section 3, routing is the planner-to-gatherer dispatch in section 2, and the evaluator-optimizer loop is the claim review in section 5. The architecture diagram and the design records live in docs/ARCHITECTURE.md in the repository.

In [ ]:
# Setup. The notebook runs from a clean clone: open_db creates the SQLite
# file and its tables on first use, and seed_from_json loads a committed
# sample run so every later section has evidence to work with, key or no
# key. The constants below are the one place to retarget the notebook; the
# relative paths assume the kernel starts in this notebook's folder, which
# is how Jupyter launches it.
import json
from contextlib import closing
from pathlib import Path

import pandas as pd

from investment_committee.config import load_settings
from investment_committee.storage import load_evidence, open_db, seed_from_json

DB_PATH = Path("../../data/final_notebook.db")
FIXTURE = Path("../../tests/fixtures/sample_evidence.json")

SEEDED_RUN_ID = json.loads(FIXTURE.read_text(encoding="utf-8"))["run"]["run_id"]

settings = load_settings()
with closing(open_db(DB_PATH)) as conn:
    inserted = seed_from_json(conn, FIXTURE)
    pool = load_evidence(conn, SEEDED_RUN_ID)

print(f"Seeded {inserted} evidence row(s); the pool holds {len(pool)}.")
pd.DataFrame([{"id": e.evidence_id, "source": e.source, "title": e.title} for e in pool])

## 2. Planner and dynamic tool use

Owner: Guna. Demonstrates: plans its research steps; uses tools dynamically; routing.

In [ ]:
# TODO(guna): plan_research on a symbol, routed gatherers filling the evidence pool.

## 3. News chain: ingest, preprocess, classify, extract, summarize

Owner: Mina. Demonstrates: prompt chaining, the classify gate, Evidence handoff.

In [ ]:
# TODO(mina): run the chain on the saved articles, show the gate and the Evidence rows.

## 4. Bull and Bear over the evidence pool

Owner: Michael. Demonstrates: model-backed drafting with citation validation.

In [ ]:
# Draft both sides from the same pool. provider defaults to the model path
# and falls back to the committed template claims when no key is set, so
# this cell runs either way; fallback_reason reports which path ran.
from investment_committee.agents.bear import draft_bear_claims_model
from investment_committee.agents.bull import draft_bull_claims_model

bull = draft_bull_claims_model(pool, settings)
bear = draft_bear_claims_model(pool, settings)

for result, side in ((bull, "bull"), (bear, "bear")):
    if result.fallback_reason:
        print(f"{side} fell back to the template: {result.fallback_reason}")
    if result.dropped_ids:
        print(f"{side} cited ids not in the pool: {result.dropped_ids}")

claims = bull.claims + bear.claims
pd.DataFrame(
    [{"side": c.side, "claim": c.text, "cites": ", ".join(c.evidence_ids)} for c in claims]
)

Both sides read the same evidence rows and pulled in opposite directions; the same news headline can appear on both sides with a different reading, which is the point of running a committee instead of one summarizer. Citations are kept exactly as drafted, including wrong ones. Catching those is the evaluator's job in the next section, not the drafting step's.

## 5. Evaluator-optimizer loop and the verdict

Owner: Michael. Demonstrates: self-reflection on output quality; the generate-evaluate-refine loop; the memo.

In [ ]:
# One full deliberation. The committee run points at the seeded run through
# parent_run_id, so it argues from the pool shown above and both this run
# and section 7's arms see identical evidence.
from investment_committee.models import Run
from investment_committee.storage import save_run
from investment_committee.workflows.evaluator_optimizer import run_committee_from_storage

committee_run = Run(symbol=pool[0].symbol, parent_run_id=SEEDED_RUN_ID)
with closing(open_db(DB_PATH)) as conn:
    save_run(conn, committee_run)
    outcome = run_committee_from_storage(conn, committee_run.run_id, settings)

print(
    f"Rounds: {outcome.rounds}; claims judged: {len(outcome.all_claims)}; "
    f"model calls: {len(outcome.model_calls)}"
)
if outcome.fallbacks:
    print(f"Degraded stages: {outcome.fallbacks}")
print()
print(outcome.memo)

The memo leads with the verdict and its reason, then lists the surviving claims with their citations. The supported counts in the diagnostics line are not the decision: the verdict comes from one chair call reading the surviving claims, so a side cannot win by writing more small claims. When a round rejects a claim, the rejected text goes back to its side with the evaluator's reason and returns as revision 1. After the verdict the run records its own confidence and open gaps as a memory note, which is the self-reflection the next run's planner reads in section 6. With no key the pass degrades to the template path and says so above.

## 6. Memory across runs

Owner: Guna with Michael. Demonstrates: learns across runs; the note written by run N read by run N+1's planner.

In [ ]:
# TODO(guna): two runs of the same symbol, show the note changing the second plan.

## 7. Evaluation and iteration: with vs without review

The comparison tests whether the review loop changes the committee's output on identical evidence. Two committee runs point at the same gathering run through parent_run_id, one with the loop on and one with it off, so the loop is the only variable. The scorecard for each arm is read back from the database rather than from objects in memory, which also demonstrates that every claim, judgment, and call was recorded.

In [ ]:
# Both arms off the seeded run, then the scorecard read back from SQLite.
from investment_committee.evaluation.harness import run_review_comparison
from investment_committee.evaluation.metrics import comparison_rows

with closing(open_db(DB_PATH)) as conn:
    comparison = run_review_comparison(conn, SEEDED_RUN_ID, settings)
    rows = comparison_rows(
        conn,
        [comparison.with_review.run.run_id, comparison.without_review.run.run_id],
    )

table = pd.DataFrame(rows).drop(columns=["run id"])
table

In [ ]:
# Supported vs rejected per arm, the comparison chart for this section.
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 3.5))
positions = range(len(table))
ax.bar([p - 0.2 for p in positions], table["supported claims"], width=0.4, label="supported")
ax.bar([p + 0.2 for p in positions], table["rejected drafts"], width=0.4, label="rejected drafts")
ax.set_xticks(list(positions))
ax.set_xticklabels([f"review {flag}" for flag in table["review loop"]])
ax.set_title(f"Claims with and without the review loop ({table['symbol'].iloc[0]})")
ax.set_xlabel("arm")
ax.set_ylabel("claims")
ax.legend()
plt.tight_layout()
plt.show()

Across our runs so far the two arms have disagreed in both directions on this pool: the reviewed arm ruled none where the unreviewed arm ruled bull on one run, and the reverse happened on another. Single verdicts vary because the model does. The chair only reads claims the evaluator supported, so without review a rejected claim is dropped, while with review it goes back to its side and returns rewritten. In our runs the reviewed arm handed the chair as many or more supported claims to weigh, and the calls and token columns price the extra round. The sweep below runs the same comparison over several symbols, gathered live.

In [ ]:
# The same comparison over several symbols, gathered live: one gathering
# run per symbol, both review arms off it. SWEEP_SYMBOLS is the one place
# to change the list. With no key the arms degrade to the template path
# and the table says so; gather warnings print per symbol.
from investment_committee.evaluation.harness import run_symbol_sweep

SWEEP_SYMBOLS = ["AAPL", "MSFT", "NVDA"]

sweep_rows = run_symbol_sweep(SWEEP_SYMBOLS, DB_PATH, settings)
sweep_table = pd.DataFrame(sweep_rows).drop(columns=["run id"])
sweep_table

The sweep is where the single-pair story either holds or falls apart, and across our runs it held in an unexpected way: the arms disagreed on every symbol, each in a different direction. The reviewed arm ruled bear on one symbol where the unreviewed arm said bull, and the reverse on another. The loop does not push verdicts toward optimism or caution. What it changes is the set of claims the chair reads: rejected claims are dropped on the unreviewed arm and rewritten on the reviewed one, and in our sweep the reviewed arm handed the chair more supported claims on every symbol. Each arm drafts on its own, so part of that gap comes from two different sets of first drafts. The calls and token columns price the extra rounds.

## 8. Conclusions and limitations

What worked, what did not, and the limitations of the system as submitted: a single market-data vendor, small evidence pools, verdicts that vary across runs on the same evidence, and the stretch items from docs/ARCHITECTURE.md that were not attempted. Owner: whole team, written at assembly.

## 9. AI tool disclosures

One first-person paragraph per author, per docs/AI_DISCLOSURE.md, plus the citation entries in the references.

## References

APA 7 entries from README.md, alphabetized, plus the AI tool entries.